# 🤖 AI-Publisher Colab Docker Kurulumu

Bu notebook, AI-Publisher platformunun tüm model konteynerlarını Google Colab'da başlatır.

### 📋 Kurulum Adımları
1. **Runtime kontrolü** - GPU durumunu doğrula
2. **Bağımlılıklar** - Drive mount ve pip paketleri
3. **Docker kurulumu** - Docker + NVIDIA Container Toolkit
4. **Repo güncelleme** - GitHub'dan en son kodları çek
5. **İmaj yükleme** - Drive'dan yükle veya sıfırdan build et
6. **Konteynerları başlat** - docker-compose ile tüm servisleri ayağa kaldır
7. **Ngrok tüneli** - Dış dünyaya aç
8. **Sunucu başlat** - colab_server.py + canlı log

> ⚠️ **CPU/GPU Ayrımı**: Whisper ve KokoroTTS CPU'da çalışır, GPU gerekmez.
> Diğer tüm modeller GPU gerektirir.

---
Her hücreyi sırasıyla çalıştırın. İlk çalıştırma sonrası oturum yeniden başlatılabilir.

## 📌 Hücre 1: Runtime Kontrolü

GPU durumunu kontrol et. Bu notebook T4 GPU ile çalışır.
CPU-only modeller (Whisper, KokoroTTS) zaten CPU kullanır.

In [ ]:
#@title Runtime Kontrolü { display-mode: "form" }
#@markdown GPU durumunu kontrol et
import subprocess
import os

def check_runtime():
    """GPU ve sistem durumunu kontrol et."""
    print("=" * 60)
    print("🔍 AI-Publisher Runtime Kontrolü")
    print("=" * 60)
    
    # GPU kontrolü
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(", ")
            print(f"✅ GPU: {gpu_info[0]}")
            print(f"   Toplam VRAM: {gpu_info[1]} MB")
            print(f"   Boş VRAM: {gpu_info[2]} MB")
        else:
            print("❌ GPU bulunamadı! Runtime > Change runtime type > GPU seçin.")
            return False
    except FileNotFoundError:
        print("❌ nvidia-smi bulunamadı. GPU runtime aktif değil.")
        return False
    
    # Disk alanı
    disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
    if disk.returncode == 0:
        lines = disk.stdout.strip().split("\n")
        if len(lines) > 1:
            parts = lines[1].split()
            print(f"💾 Disk: {parts[2]} kullanıldı / {parts[1]} toplam")
    
    # RAM
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_kb = int(line.split()[1])
                print(f"🧠 RAM: {total_kb // 1024} MB")
                break
    
    print("\n✅ Runtime hazır. Sonraki hücreye geçin.")
    return True

check_runtime()

## 📌 Hücre 2: Bağımlılıklar ve Drive Mount

Google Drive'ı bağla ve gerekli Python paketlerini kur.
Docker imajları Drive'da önbelleklenir, her seferinde yeniden build edilmez.

In [ ]:
#@title Bağımlılıklar ve Drive Mount { display-mode: "form" }
import subprocess
import sys
import os
import time

print("📦 Bağımlılıklar kuruluyor...")

# Google Drive mount
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("✅ Google Drive bağlandı.")
    else:
        print("✅ Google Drive zaten bağlı.")
except Exception as e:
    print(f"⚠️ Drive mount hatası: {e}")

# Python bağımlılıkları
deps = [
    "flask",
    "requests",
    "pyngrok",
    "opencv-python-headless",
    "numpy<2.0.0",
    "yt-dlp",
]

for pkg in deps:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True
    )

print("✅ Python bağımlılıkları kuruldu.")

# Drive çıktı dizinini oluştur
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Çıktı dizini hazır: {OUTPUT_DIR}")

print("\n✅ Hücre 2 tamamlandı.")

## 📌 Hücre 3: Docker ve NVIDIA Container Toolkit

Docker ve GPU desteği için NVIDIA Container Toolkit kurulur.
Bu işlem sadece ilk çalıştırmada yapılır.

In [ ]:
#@title Docker + NVIDIA Toolkit Kurulumu { display-mode: "form" }
import subprocess
import shutil
import sys

def run(cmd, label="", retries=3):
    """Komutu çalıştır, hata olursa tekrar dene."""
    for attempt in range(1, retries + 1):
        try:
            result = subprocess.run(
                cmd, shell=True, check=True,
                capture_output=True, text=True, timeout=300
            )
            return True
        except Exception as e:
            if attempt == retries:
                print(f"  ❌ {label} başarısız ({retries} deneme): {e}")
                return False
            time.sleep(3)
    return False

# Docker zaten kurulu mu?
if shutil.which("docker"):
    print("✅ Docker zaten kurulu.")
else:
    print("🐳 Docker kuruluyor...")
    run("apt-get update -q", label="apt update")
    run("apt-get install -y -q docker.io", label="Docker.io")
    run("service docker start", label="Docker başlatma")
    print("✅ Docker kuruldu.")

# NVIDIA Container Toolkit
nvidia_keyring = "/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg"
if os.path.exists(nvidia_keyring):
    print("✅ NVIDIA Container Toolkit zaten kurulu.")
else:
    print("🔧 NVIDIA Container Toolkit kuruluyor...")
    run(
        'curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | '
        'gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg',
        label="NVIDIA GPG anahtarı"
    )
    run(
        'curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | '
        'sed "s#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g" | '
        'tee /etc/apt/sources.list.d/nvidia-container-toolkit.list',
        label="NVIDIA repo"
    )
    run("apt-get update -q", label="apt update")
    run("apt-get install -y -q nvidia-container-toolkit", label="NVIDIA toolkit")
    run("nvidia-ctk runtime configure --runtime=docker", label="NVIDIA docker config")
    run("service docker restart", label="Docker yeniden başlatma")
    print("✅ NVIDIA Container Toolkit kuruldu.")

# Docker çalışıyorsa GPU'yu doğrula
gpu_test = subprocess.run(
    "docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi --query-gpu=name --format=csv,noheader",
    shell=True, capture_output=True, text=True, timeout=120
)
if gpu_test.returncode == 0:
    gpu_name = gpu_test.stdout.strip()
    print(f"✅ Docker GPU doğrulandı: {gpu_name}")
else:
    print(f"⚠️ Docker GPU testi başarısız: {gpu_test.stderr[:200]}")

print("\n✅ Hücre 3 tamamlandı.")

## 📌 Hücre 4: Repo Güncelleme

GitHub'dan en son kodları çek veya mevcut repo'yu güncelle.

In [ ]:
#@title Repo Güncelle { display-mode: "form" }
import subprocess
import os

REPO_URL = "https://github.com/Arda-Avci/AI-Publisher.git"
REPO_DIR = "/content/AI-Publisher"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("🔄 Mevcut repo güncelleniyor...")
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "origin", "main"],
        capture_output=True, text=True
    )
    print("✅ Repo güncellendi.")
else:
    print("📥 Repo klonlanıyor...")
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print("✅ Repo klonlandı.")

# colab_docker dizinini kontrol et
docker_dir = os.path.join(REPO_DIR, "colab_docker")
if os.path.exists(docker_dir):
    models = [d for d in os.listdir(docker_dir) if os.path.isdir(os.path.join(docker_dir, d))]
    print(f"\n📁 Bulunan konteyner dizinleri ({len(models)}):")
    for m in sorted(models):
        print(f"   • {m}")
else:
    print("❌ colab_docker dizini bulunamadı!")

print("\n✅ Hücre 4 tamamlandı.")

## 📌 Hücre 5: Docker İmaj Yükleme

İmajları Google Drive'dan yükle veya sıfırdan build et.
Build edilen imajlar Drive'a kaydedilir, bir sonraki sefer yüklenir.

> 🕐 İlk build ~30-45 dk sürebilir. Sonraki çalıştırmalar ~5 dk.

In [ ]:
#@title Docker İmajlarını Yükle veya Build Et { display-mode: "form" }
import subprocess
import os
import sys
import time

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
REPO_DIR = "/content/AI-Publisher"
BUILD_DIR = os.path.join(REPO_DIR, "colab_docker")

# Tüm modeller (build_all.sh sırasıyla)
ALL_MODELS = [
    "cogvideox", "wan", "ltx", "hunyuan",
    "xtts",
    "audioldm2",
    "wav2lip", "musetalk",
    "whisper",
    "stablediffusion",
    "kokorotts",
]

os.makedirs(DRIVE_IMAGES_DIR, exist_ok=True)

# İmaj yükleme fonksiyonu
def load_image(model_name):
    """Drive'dan imaj yükle veya build et."""
    image_tag = f"ai-publisher-{model_name}:latest"
    
    # İmaj zaten yüklü mü?
    check = subprocess.run(
        ["docker", "images", "-q", image_tag],
        capture_output=True, text=True
    )
    if check.stdout.strip():
        print(f"  ✅ {model_name} zaten yüklü.")
        return True
    
    # Drive'dan yükle
    tar_path = os.path.join(DRIVE_IMAGES_DIR, f"{model_name}.tar.gz")
    if os.path.exists(tar_path):
        print(f"  📦 {model_name} Drive'dan yükleniyor...")
        result = subprocess.run(
            ["docker", "load", "-i", tar_path],
            capture_output=True, text=True, timeout=600
        )
        if result.returncode == 0:
            print(f"  ✅ {model_name} Drive'dan yüklendi.")
            return True
        else:
            print(f"  ⚠️ {model_name} Drive yükleme hatası, build edilecek.")
    
    # Build et
    model_dir = os.path.join(BUILD_DIR, model_name)
    dockerfile = os.path.join(model_dir, "Dockerfile")
    if not os.path.exists(dockerfile):
        print(f"  ❌ {model_name}/Dockerfile bulunamadı!")
        return False
    
    print(f"  🔨 {model_name} build ediliyor...")
    result = subprocess.run(
        ["docker", "build", "-t", image_tag, model_dir],
        capture_output=True, text=True, timeout=3600
    )
    if result.returncode != 0:
        print(f"  ❌ {model_name} build hatası:")
        print(result.stderr[-500:])
        return False
    
    # Drive'a kaydet
    print(f"  💾 {model_name} Drive'a kaydediliyor...")
    subprocess.run(
        ["docker", "save", image_tag, "-o", tar_path],
        capture_output=True, timeout=600
    )
    print(f"  ✅ {model_name} build ve kaydedildi.")
    return True

# Base imajı kontrol et
print("🔍 Base imaj kontrolü...")
base_check = subprocess.run(
    ["docker", "images", "-q", "ai-publisher-base:latest"],
    capture_output=True, text=True
)
if not base_check.stdout.strip():
    print("📦 Base imaj build ediliyor...")
    base_tar = os.path.join(DRIVE_IMAGES_DIR, "base.tar.gz")
    if os.path.exists(base_tar):
        subprocess.run(["docker", "load", "-i", base_tar], capture_output=True, timeout=600)
    else:
        base_dockerfile = os.path.join(BUILD_DIR, "Dockerfile.base")
        subprocess.run(
            ["docker", "build", "-t", "ai-publisher-base:latest", "-f", base_dockerfile, BUILD_DIR],
            capture_output=True, text=True, timeout=1200
        )
        subprocess.run(
            ["docker", "save", "ai-publisher-base:latest", "-o", base_tar],
            capture_output=True, timeout=600
        )
    print("✅ Base imaj hazır.")
else:
    print("✅ Base imaj zaten yüklü.")

# Tüm modelleri yükle
print("\n📦 Konteyner imajları yükleniyor...")
start_time = time.time()
results = {}
for model in ALL_MODELS:
    print(f"\n[{ALL_MODELS.index(model)+1}/{len(ALL_MODELS)}] {model}")
    results[model] = load_image(model)

elapsed = time.time() - start_time
success = sum(1 for v in results.values() if v)
print(f"\n{'=' * 50}")
print(f"📊 Sonuç: {success}/{len(ALL_MODELS)} imaj hazır ({elapsed:.0f}s)")

failed = [m for m, ok in results.items() if not ok]
if failed:
    print(f"❌ Başarısız: {', '.join(failed)}")
    print("Bu modeller çalışmayacak. Dockerfile'ları kontrol edin.")
else:
    print("✅ Tüm imajlar hazır!")

print("\n✅ Hücre 5 tamamlandı.")

## 📌 Hücre 6: Konteynerları Başlat

docker-compose ile tüm servisleri başlatır.

### 🖥️ CPU/GPU Dağılımı
| Konteyner | Mod | Port |
|:--|:--|:--|
| cogvideox, wan, ltx, hunyuan | GPU | 5001, 5008, 5009, 5010 |
| xtts, audioldm2 | GPU | 5002, 5003 |
| wav2lip, musetalk, stablediffusion | GPU | 5004, 5005, 5007 |
| **whisper, kokorotts** | **CPU** | 5006, 5011 |

In [ ]:
#@title Konteynerları Başlat { display-mode: "form" }
import subprocess
import os
import time

BUILD_DIR = "/content/AI-Publisher/colab_docker"
SHARED_DIR = os.path.join(BUILD_DIR, "shared")
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/outputs"

# shared ve outputs dizinlerini hazırla
os.makedirs(SHARED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Eski konteynerları durdur (varsa)
print("🧹 Eski konteynerlar temizleniyor...")
subprocess.run(
    ["docker", "compose", "-f", os.path.join(BUILD_DIR, "docker-compose.yml"), "down"],
    capture_output=True, cwd=BUILD_DIR
)
time.sleep(3)

# docker-compose ile başlat
print("🚀 Konteynerlar başlatılıyor...")
result = subprocess.run(
    ["docker", "compose", "-f", "docker-compose.yml", "up", "-d"],
    capture_output=True, text=True, cwd=BUILD_DIR, timeout=120
)

if result.returncode != 0:
    print(f"❌ docker-compose hatası:")
    print(result.stderr[-1000:])
else:
    print("✅ Konteynerlar başlatıldı.")

# Durum kontrolü
time.sleep(10)
print("\n📊 Konteyner durumları:")
status = subprocess.run(
    ["docker", "compose", "-f", "docker-compose.yml", "ps"],
    capture_output=True, text=True, cwd=BUILD_DIR
)
print(status.stdout)

print("\n✅ Hücre 6 tamamlandı.")

## 📌 Hücre 7: Ngrok Tüneli

Colab'ı dış dünyaya açar. Ngrok token'ınız gerekir.
> 🔑 [ngrok.com](https://ngrok.com) adresinden ücretsiz token alabilirsiniz.

In [ ]:
#@title Ngrok Tüneli Kur { display-mode: "form" }
#@markdown Ngrok Auth Token'ınızı girin
NGROK_TOKEN = "" #@param {type:"string"}

import os
import subprocess
import time

if not NGROK_TOKEN:
    # Colab userdata'dan dene
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get("NGROK_TOKEN")
    except:
        pass

if not NGROK_TOKEN:
    print("❌ NGROK_TOKEN gerekli! Hücreye yapıştırın veya Colab Secrets'a ekleyin.")
    print("   https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    # Eski ngrok süreçlerini temizle
    subprocess.run("pkill -9 ngrok", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    
    # Ngrok config ve başlat
    os.makedirs(os.path.expanduser("~/.ngrok2"), exist_ok=True)
    with open(os.path.expanduser("~/.ngrok2/ngrok.yml"), "w") as f:
        f.write(f"authtoken: {NGROK_TOKEN}\n")
    
    # Ngrok tünelini başlat (port 5000 = colab_server.py)
    subprocess.Popen(
        ["ngrok", "http", "5000", "--log=stdout"],
        stdout=open("/content/ngrok.log", "w"),
        stderr=subprocess.STDOUT
    )
    
    # URL bekle
    print("⏳ Ngrok URL bekleniyor...")
    ngrok_url = None
    for _ in range(30):
        time.sleep(2)
        try:
            import requests
            resp = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=5)
            data = resp.json()
            if data.get("tunnels"):
                ngrok_url = data["tunnels"][0]["public_url"]
                break
        except:
            pass
    
    if ngrok_url:
        print(f"\n🔗 NGROK URL (Node.js projenize yapıştırın):")
        print(f"\n   {ngrok_url}\n")
        # URL'yi dosyaya kaydet
        with open("/content/ngrok_url.txt", "w") as f:
            f.write(ngrok_url)
    else:
        print("❌ Ngrok URL alınamadı. Token'ı kontrol edin.")
        print("   Logs: /content/ngrok.log")

print("\n✅ Hücre 7 tamamlandı.")

## 📌 Hücre 8: Sunucu Başlat ve Canlı Log

colab_server.py'yi başlatır ve canlı log akışını gösterir.
Bu sunucu Node.js istemciden gelen istekleri alır.

In [ ]:
#@title Sunucu Başlat { display-mode: "form" }
import subprocess
import os
import sys
import time

REPO_DIR = "/content/AI-Publisher"
SERVER_SCRIPT = os.path.join(REPO_DIR, "colab_server.py")
LOG_FILE = "/content/colab_server.log"

# colab_server.py mevcut mu?
if not os.path.exists(SERVER_SCRIPT):
    print(f"❌ colab_server.py bulunamadı: {SERVER_SCRIPT}")
else:
    # Eski süreci durdur
    subprocess.run("pkill -f colab_server.py", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    
    # Ngrok URL'sini al
    ngrok_url = ""
    if os.path.exists("/content/ngrok_url.txt"):
        with open("/content/ngrok_url.txt") as f:
            ngrok_url = f.read().strip()
    
    # Sunucu env
    env = os.environ.copy()
    env["NGROK_URL"] = ngrok_url
    
    # colab_server.py'yi başlat
    print("🚀 colab_server.py başlatılıyor...")
    with open(LOG_FILE, "w") as log:
        subprocess.Popen(
            [sys.executable, "-u", SERVER_SCRIPT],
            stdout=log,
            stderr=subprocess.STDOUT,
            env=env,
        )
    
    # Başlatılmasını bekle
    time.sleep(5)
    if os.path.exists(LOG_FILE) and os.path.getsize(LOG_FILE) > 0:
        print("✅ Sunucu başlatıldı.")
        print("\n📋 Son log satırları:")
        with open(LOG_FILE) as f:
            lines = f.readlines()
            for line in lines[-20:]:
                print(f"   {line.rstrip()}")
    else:
        print("⚠️ Sunucu henüz log üretmedi. Biraz bekleyin.")

print("\n✅ Hücre 8 tamamlandı.")
print("\n" + "=" * 60)
print("🎉 KURULUM TAMAMLANDI!")
print("=" * 60)

## 📌 Hücre 9: Canlı Log İzleme (İsteğe Bağlı)

Sunucu loglarını canlı olarak izlemek için bu hücreyi çalıştırın.
Clear output ile durdurabilirsiniz.

In [ ]:
#@title Canlı Log İzleme { display-mode: "form" }
#@markdown Clear output ile durdurun
import time
import os

LOG_FILE = "/content/colab_server.log"

if not os.path.exists(LOG_FILE):
    print("❌ Log dosyası bulunamadı. Önce Hücre 8'i çalıştırın.")
else:
    print("📋 Canlı loglar (Clear output ile durdurun):")
    print("-" * 60)
    with open(LOG_FILE) as f:
        # Dosya sonuna git
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line.rstrip())
            else:
                time.sleep(1)